# 🎤 TTS Generator + Auto-Upload to Website

**Microsoft Edge TTS + SFTP Auto-Upload**

## ✅ Features:
- **NO TEXT LENGTH LIMIT** - any text length works
- **16 English voices** (US, UK, Australian, Indian, Canadian)
- **Speed control** (0.5x to 2.0x)
- **Auto-upload to website** via SFTP
- **Custom filename** support
- **Audio preview** in Colab
- **Direct link** to uploaded file
- **Clear buttons** for text and filename
- **Quick link** to TTS player on website

## 🔐 Setup

Before running, add SFTP password to Colab Secrets:
1. Click 🔑 (key icon) in the left panel of Colab
2. Add new secret with name: `SFTP_PASS`
3. Value: your SFTP password
4. Enable "Notebook access"

## 🚀 One-click deployment

In [ ]:
# ============================================================================
# 📦 INSTALL DEPENDENCIES
# ============================================================================
print("=" * 80)
print("📦 INSTALLING DEPENDENCIES")
print("=" * 80)

!pip install edge-tts ipywidgets nest_asyncio paramiko -q

print("✅ Dependencies installed")

In [ ]:
# ============================================================================
# 🚀 TTS GENERATOR + AUTO-UPLOAD TO WEBSITE
# ============================================================================
print("\n" + "=" * 80)
print("🚀 LAUNCHING TTS GENERATOR")
print("=" * 80)

import edge_tts
import asyncio
import re
import time
import os
import paramiko
from IPython.display import Audio, display, HTML
import ipywidgets as widgets
import nest_asyncio

nest_asyncio.apply()

print("✅ Modules imported")

# ============================================================================
# 🔐 SFTP CONFIGURATION (PASSWORD FROM COLAB SECRETS)
# ============================================================================
# Password is stored in Colab Secrets as SFTP_PASS
# To add it: click 🔑 icon in left panel → Add new secret → SFTP_PASS

from google.colab import userdata

SFTP_HOST = "185.182.65.58"
SFTP_PORT = 22
SFTP_USER = "root"
SFTP_PASS = userdata.get('SFTP_PASS')  # ← FROM COLAB SECRETS
REMOTE_PATH = "/var/www/lepka-iz-plastilina.ru/public_html/tts/alec/"

def upload_to_site(local_file):
    """Upload file to website via SFTP"""
    try:
        transport = paramiko.Transport((SFTP_HOST, SFTP_PORT))
        transport.connect(username=SFTP_USER, password=SFTP_PASS)
        sftp = paramiko.SFTPClient.from_transport(transport)
        
        remote_file = os.path.join(REMOTE_PATH, os.path.basename(local_file))
        sftp.put(local_file, remote_file)
        
        sftp.close()
        transport.close()
        
        return f"✅ File uploaded: https://lepka-iz-plastilina.ru.net/tts/alec/{os.path.basename(local_file)}"
    except Exception as e:
        return f"❌ Upload error: {e}"

print("✅ SFTP configured (password from Colab Secrets)")

# ============================================================================
# 🎤 VOICES
# ============================================================================
voices = {
    "us_young": "en-US-ChristopherNeural",
    "us_female": "en-US-JennyNeural",
    "us_male": "en-US-EricNeural",
    "uk_female": "en-GB-SoniaNeural",
    "uk_male": "en-GB-RyanNeural",
    "au_female": "en-AU-NatashaNeural",
    "au_male": "en-AU-WilliamNeural",
    "indian_female": "en-IN-NeerjaNeural",
    "indian_male": "en-IN-PrabhatNeural",
    "ca_female": "en-CA-ClaraNeural",
    "ca_male": "en-CA-LiamNeural",
}

print(f"✅ {len(voices)} voices loaded")

# ============================================================================
# 🎵 AUDIO GENERATION FUNCTION
# ============================================================================
async def generate_audio_async(text, voice, speed):
    voice_id = voices.get(voice, "en-US-JennyNeural")
    speed_float = float(speed)
    rate = f"+{int((speed_float - 1.0) * 100)}%" if speed_float != 1.0 else "+0%"
    
    communicate = edge_tts.Communicate(text, voice_id, rate=rate)
    audio_data = b""
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            audio_data += chunk["data"]
    return audio_data

def generate_audio_sync(text, voice, speed):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            return loop.run_until_complete(generate_audio_async(text, voice, speed))
        else:
            return asyncio.run(generate_audio_async(text, voice, speed))
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(generate_audio_async(text, voice, speed))
        finally:
            loop.close()

print("✅ Audio generator ready")

# ============================================================================
# 📝 TEXT CLEANING
# ============================================================================
def clean_text(text):
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

print("✅ Text cleaner ready")

# ============================================================================
# 🎛️ CREATE INTERFACE
# ============================================================================
print("\n" + "=" * 80)
print("📝 CREATING INTERFACE")
print("=" * 80 + "\n")

# Text input
text_input = widgets.Textarea(
    value='Hello, this is a test of the TTS generator!',
    placeholder='Enter text for voice synthesis...',
    description='📝 Text:',
    layout=widgets.Layout(width='100%', height='150px')
)

# Filename input
filename_input = widgets.Text(
    value='my_audio',
    placeholder='Enter filename (without extension)',
    description='📁 Filename:',
    layout=widgets.Layout(width='60%')
)

# Voice dropdown (default: us_young)
voice_dropdown = widgets.Dropdown(
    options=list(voices.keys()),
    value='us_young',
    description='🎙️ Voice:',
    layout=widgets.Layout(width='50%')
)

# Speed slider
speed_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=2.0,
    step=0.1,
    description='⚡ Speed:',
    layout=widgets.Layout(width='50%')
)

# Status label
status_label = widgets.Label(value='⏳ Waiting...')

# Output areas
audio_output = widgets.Output()
upload_output = widgets.Output()

# ============================================================================
# 🗑️ CLEAR BUTTONS
# ============================================================================
def clear_text(b):
    text_input.value = ''
    status_label.value = '🗑️ Text cleared'

def clear_filename(b):
    filename_input.value = ''
    status_label.value = '🗑️ Filename cleared'

def clear_all(b):
    text_input.value = ''
    filename_input.value = ''
    status_label.value = '🗑️ All cleared'

clear_text_btn = widgets.Button(
    description='🗑️ Clear Text',
    button_style='warning',
    layout=widgets.Layout(width='auto')
)
clear_text_btn.on_click(clear_text)

clear_filename_btn = widgets.Button(
    description='🗑️ Clear Filename',
    button_style='warning',
    layout=widgets.Layout(width='auto')
)
clear_filename_btn.on_click(clear_filename)

clear_all_btn = widgets.Button(
    description='🗑️ Clear All',
    button_style='danger',
    layout=widgets.Layout(width='auto')
)
clear_all_btn.on_click(clear_all)

# ============================================================================
# 🔗 LINK TO WEBSITE
# ============================================================================
website_link = widgets.HTML(
    value='<a href="https://lepka-iz-plastilina.ru.net/tts/tts_player.html" target="_blank" style="font-size:16px; background:#4CAF50; color:white; padding:10px 20px; border-radius:5px; text-decoration:none; display:inline-block;">🌐 Open TTS Player on Website</a>'
)

# ============================================================================
# 🚀 GENERATE FUNCTION
# ============================================================================
def on_generate_clicked(b):
    raw_text = text_input.value
    text = clean_text(raw_text)
    
    if not text:
        status_label.value = '❌ Please enter text!'
        return
    
    filename = filename_input.value.strip() or 'my_audio'
    filename = re.sub(r'\.mp3$', '', filename)
    filename = re.sub(r'[^a-zA-Z0-9_\-. ]', '_', filename)
    if not filename:
        filename = 'my_audio'
    
    status_label.value = '⏳ Generating audio...'
    audio_output.clear_output()
    upload_output.clear_output()
    
    with audio_output:
        try:
            audio_data = generate_audio_sync(text, voice_dropdown.value, speed_slider.value)
            
            if not audio_data:
                raise Exception("Audio is empty")
            
            full_filename = f"{filename}.mp3"
            
            # Save locally
            with open(full_filename, 'wb') as f:
                f.write(audio_data)
            
            # Show audio player
            display(Audio(data=audio_data, autoplay=False))
            
            # Upload to website via SFTP
            status_label.value = '⏳ Uploading to website...'
            result = upload_to_site(full_filename)
            
            with upload_output:
                print(result)
                print(f"\n🔗 File URL: https://lepka-iz-plastilina.ru.net/tts/alec/{full_filename}")
                print(f"\n🌐 Open TTS Player: https://lepka-iz-plastilina.ru.net/tts/tts_player.html")
            
            status_label.value = f'✅ Done! {full_filename} ({round(len(audio_data)/1024, 1)} KB)'
            
        except Exception as e:
            status_label.value = f'❌ Error: {e}'

# Generate button
generate_button = widgets.Button(
    description='🎵 GENERATE AND UPLOAD',
    button_style='success',
    layout=widgets.Layout(width='100%')
)
generate_button.on_click(on_generate_clicked)

# ============================================================================
# 📊 DISPLAY INTERFACE
# ============================================================================
display(text_input)
display(widgets.HBox([clear_text_btn, clear_all_btn]))
display(filename_input)
display(widgets.HBox([clear_filename_btn]))
display(widgets.HBox([voice_dropdown, speed_slider]))
display(generate_button)
display(website_link)
display(status_label)
display(audio_output)
display(upload_output)

print("\n" + "=" * 80)
print("💡 INSTRUCTIONS:")
print("=" * 80)
print("1. Enter text in the field above")
print("2. Choose voice (default: us_young)")
print("3. Enter filename (without .mp3)")
print("4. Click 'GENERATE AND UPLOAD'")
print("5. Audio will be generated and uploaded to:")
print("   https://lepka-iz-plastilina.ru.net/tts/alec/")
print("\n🔗 After upload, click 'Open TTS Player' to listen on website")
print("\n🗑️ Use Clear buttons to reset fields")
print("=" * 80)